# AI Image Generator Backend (Automatic1111 API)
Run this notebook to start your free cloud GPU backend. It will provide a public Gradio URL that you will paste into your custom Web App.

In [ ]:
import os
import shutil

# Fix Colab git authentication issues
!git config --global credential.helper ""
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["GIT_ASKPASS"] = ""

# Fix Matplotlib crash caused by Colab's default inline backend
if "MPLBACKEND" in os.environ:
    del os.environ["MPLBACKEND"]

# Stability-AI DELETED their stable diffusion repo! The CompVis fallback we tried lacked the 'midas' module for SD 2.0.
# We MUST use a 1:1 identical fork of the deleted Stability-AI repo that still exists on GitHub.
os.environ["STABLE_DIFFUSION_REPO"] = "https://github.com/nlile/stablediffusion.git"
os.environ["STABLE_DIFFUSION_COMMIT_HASH"] = "47b6b607fdd31875c9279cd2f4f16b92e4ea958e"

# 1. Install Python 3.10 to ensure we have pre-compiled wheels for all packages
!add-apt-repository -y ppa:deadsnakes/ppa
!apt-get update
!apt-get install -y python3.10 python3.10-venv python3.10-dev

if not os.path.exists('/content/stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui

%cd /content/stable-diffusion-webui

# FIX: Delete the broken repo cloned from previous runs
if os.path.exists('repositories/stable-diffusion-stability-ai'):
    shutil.rmtree('repositories/stable-diffusion-stability-ai')

# Revert any of our previous patches to get back to a clean state
!git checkout requirements.txt requirements_versions.txt 2>/dev/null || true

# Create the Python 3.10 virtual environment
if not os.path.exists('venv'):
    !python3.10 -m venv venv

models = [
    {"file": "juggernaut_xl.safetensors", "url": "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"},
    {"file": "realvis_xl_v4.safetensors", "url": "https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors"}
]
for model in models:
    if not os.path.exists(f'models/Stable-diffusion/{model["file"]}'):
        print(f'Downloading {model["file"]}...')
        !wget -q --show-progress -O models/Stable-diffusion/{model["file"]} {model["url"]}

# 3. Launch using Python 3.10 (Using default REQS_FILE now that Python 3.10 fixes the PIP backtracking)
!COMMANDLINE_ARGS="--api --share --opt-sdp-attention --cors-allow-origins=* --enable-insecure-extension-access --gradio-queue" ./venv/bin/python launch.py